In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_ori = pd.read_csv('cleaned.csv')
df = df_ori.copy()
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib.pyplot as plt
import seaborn as sns

# === Define Features ===
numerical_features = ['ENGINESIZE', 'CYLINDERS', 'FUELCONSUMPTION_COMB', 'CO2EMISSIONS']
categorical_features = ['VEHICLECLASS', 'TRANSMISSION', 'FUEL_TYPE']

# === Encode Categorical Features ===
df_encoded = df.copy()
label_encoders = {}
for feature in categorical_features:
    le = LabelEncoder()
    df_encoded[feature] = le.fit_transform(df[feature])
    label_encoders[feature] = le

# === Combine Features ===
X_features = df_encoded[numerical_features + categorical_features]

# === Standardize ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# === PCA to 2D ===
pca = PCA(n_components=2)
X_2D = pca.fit_transform(X_scaled)

# === KMeans Clustering ===
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10) # 2 is better than 3 on kmeans
clusters = kmeans.fit_predict(X_2D)

labels = kmeans.labels_

# Calculate clustering performance metrics
silhouette_avg = silhouette_score(X_2D, labels)
davies_bouldin_avg = davies_bouldin_score(X_2D, labels)
calinski_harabasz_avg = calinski_harabasz_score(X_2D, labels)

print(f"Silhouette Score: {silhouette_avg:.3f}") # closer to 1 the better
print(f"Davies-Bouldin Index: {davies_bouldin_avg:.3f}") # the lower the better
print(f"Calinski-Harabasz Index: {calinski_harabasz_avg:.3f}") # the higher the better

# === Visualization ===
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_2D[:, 0], X_2D[:, 1], c=clusters, cmap='viridis', alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('K-Means Clustering on PCA Components\n(Numerical + Categorical Features)')
plt.colorbar(scatter, label='Cluster')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            c='red', marker='x', s=200, linewidths=3, label='Centroids')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# === Append Cluster Labels ===
df['Cluster'] = clusters

# === Enhanced Cluster Summary ===
print(f"\n=== ENHANCED CLUSTER SUMMARY ===")
for cluster in range(kmeans.n_clusters):
    cluster_data = df[df['Cluster'] == cluster]
    print(f"\nCluster {cluster} ({len(cluster_data)} vehicles):")
    print(f"  Average Engine Size: {cluster_data['ENGINESIZE'].mean():.1f}L")
    print(f"  Average CO2 Emissions: {cluster_data['CO2EMISSIONS'].mean():.0f}g/km")
    print(f"  Average Fuel Consumption: {cluster_data['FUELCONSUMPTION_COMB'].mean():.1f}L/100km")
    
    # Show top 3 makers in the cluster with their top models
    top_makers = cluster_data['MAKER'].value_counts().head(3)
    print("  Top Makers and Their Models:")
    
    for i, (maker, count) in enumerate(top_makers.items(), 1):
        maker_data = cluster_data[cluster_data['MAKER'] == maker]
        top_models = maker_data['MODEL'].value_counts().head(3)
        
        print(f"    {i}. {maker}: {count} vehicles")
        for j, (model, model_count) in enumerate(top_models.items(), 1):
            print(f"       {j}. {model}: {model_count} vehicles")

print(f"\nTotal vehicles clustered: {len(df)}")

# === Optional: Visualize MAKER distribution per cluster ===
top_makers_overall = df['MAKER'].value_counts().head(8).index
subset_df = df[df['MAKER'].isin(top_makers_overall)]

plt.figure(figsize=(12, 6))
sns.countplot(data=subset_df, x='MAKER', hue='Cluster', palette='Set2')
plt.title("Top Car Makers by Cluster")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# === Additional Visualization: Top Models per Cluster ===
print(f"\n=== TOP MODELS PER CLUSTER ===")
for cluster in range(kmeans.n_clusters):
    cluster_data = df[df['Cluster'] == cluster]
    print(f"\nCluster {cluster} - Top 5 Models Overall:")
    top_models_cluster = cluster_data['MODEL'].value_counts().head(5)
    for i, (model, count) in enumerate(top_models_cluster.items(), 1):
        maker = cluster_data[cluster_data['MODEL'] == model]['MAKER'].iloc[0]
        print(f"  {i}. {maker} {model}: {count} vehicles")

# === Create a detailed breakdown visualization ===
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for cluster in range(kmeans.n_clusters):
    cluster_data = df[df['Cluster'] == cluster]
    top_models = cluster_data.groupby(['MAKER', 'MODEL']).size().reset_index(name='count')
    top_models = top_models.nlargest(10, 'count')
    top_models['maker_model'] = top_models['MAKER'] + ' ' + top_models['MODEL']
    
    axes[cluster].barh(range(len(top_models)), top_models['count'])
    axes[cluster].set_yticks(range(len(top_models)))
    axes[cluster].set_yticklabels(top_models['maker_model'], fontsize=8)
    axes[cluster].set_xlabel('Number of Vehicles')
    axes[cluster].set_title(f'Cluster {cluster} - Top 10 Models')
    axes[cluster].invert_yaxis()

plt.tight_layout()
plt.show()